<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

# Maestría en Inteligencia Artificial  
## Proyecto de Innovación III  
### Detección de pistolas y cuchillos en actos delictivos mediante visión computacional
---


**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/Weapon-Detection/blob/main/notebooks/01_eda.ipynb.ipynb)



# Introducción

Este notebook utiliza [Ultralytics](https://docs.ultralytics.com/) para entrenar modelos de detección de objetos YOLO11 con un conjunto de datos personalizado. Al finalizar este Colab, tendremos un modelo YOLO entrenado específicamente para la detección de armas.

# Descripción del Dataset

El conjunto de datos utilizado en este proyecto proviene de [DaSCI (Instituto Andaluz Interuniversitario en Data Science and Computational Intelligence)](https://dasci.es/opendata/deteccion-de-armas-open-data/), específicamente de su sección “Detección de Armas – Open Data”.


**¿Qué contiene este dataset?**

DaSCI pone a disposición varios sub-conjuntos de datos orientados a la investigación en detección automática de armas a partir de imágenes, con el fin de entrenar modelos de Deep Learning para videovigilancia inteligente.


**Tiene dos grandes categorías:**



*   Clasificación de imágenes, donde las imágenes están organizadas en directorios por clase (“pistola”, “cuchillo”, etc.).


*   Detección de objetos, con anotaciones en formato Pascal VOC (XML), para que los modelos puedan aprender a localizar armas en la escena.


## Sub-conjuntos más relevantes




1.   **Detección de pistolas (Pistol Detection)**

Aproximadamente 3.000 imágenes con pistolas, en escenarios variados como videovigilancia y fondos complejos.


Todas las imágenes tienen anotaciones en formato Pascal VOC, lo que permite entrenar modelos de detección.


2.   **Detección de armas blancas (Knife Detection)**

El dataset contiene 2.078 imágenes donde aparece al menos un cuchillo.


Incluye diferentes tipos de cuchillos, distancias, oclusión y contexto, lo que lo hace robusto para tareas reales.


3.   **Sohas Weapon Detection**

Incluye seis clases de “Sohas Weapon” para la detección: pistola, cuchillo, billete, cartera, teléfono y tarjeta.


**Configuración de entorno (Google Colab o Local)**

In [ ]:
import os

# Detectar entorno
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

# Definir base
if IN_COLAB:
    BASE_PATH = "/content"
else:
    BASE_PATH = os.getcwd()

print(f"BASE_PATH: {BASE_PATH}")

In [ ]:
from google.colab import userdata

token = userdata.get('KAGGLE_TOKEN')

# Definir la ruta del directorio y el archivo
kaggle_dir = '/root/.kaggle'
token_file = os.path.join(kaggle_dir, 'access_token')

# 3rear el directorio si no existe
os.makedirs(kaggle_dir, exist_ok=True)

# Escribir el token en el archivo
with open(token_file, 'w') as f:
    f.write(token)

# Cambiar permisos para mayor seguridad
os.chmod(token_file, 0o600)

print("Configuración completada exitosamente.")

In [ ]:
zip_filename = "WeaponDetection.zip"
zip_path = f"{BASE_PATH}/{zip_filename}"

if not os.path.exists(zip_path):
    print("Descargando dataset desde Kaggle...")
    !kaggle datasets download -d sebaxtian92/WeaponDetection -p {BASE_PATH}
else:
    print("El ZIP ya existe, no se descarga de nuevo.")

**Importación de librerías requeridas**

Descarga dependencias y las instala automáticamente en Colab.

In [ ]:
if IN_COLAB:
    !wget -q https://raw.githubusercontent.com/sebastianb92/Weapon-Detection/main/requirements.txt -O requirements.txt
    !uv pip install -r requirements.txt
else:
    print("Entorno local — instala con: pip install -r requirements.txt")

In [ ]:
# === Análisis y manipulación de datos ===
import numpy as np              # Operaciones numéricas con arrays
import pandas as pd             # Manejo de dataframes para análisis de anotaciones

# === Visualización ===
import matplotlib.pyplot as plt # Creación de gráficos y visualizaciones
import seaborn as sns           # Visualizaciones estadísticas mejoradas

# === Procesamiento de imágenes ===
import cv2                      # Lectura de imágenes y dibujo de bounding boxes
from PIL import Image           # Carga de imágenes y obtención de dimensiones

# === Utilidades ===
from tqdm.auto import tqdm      # Barras de progreso para bucles largos
from pathlib import Path        # Manejo moderno de rutas de archivos
from glob import glob           # Búsqueda de archivos con patrones
import xml.etree.ElementTree as ET  # Parseo de anotaciones en formato PASCAL VOC
import zipfile                  # Descompresión del dataset
import os                       # Operaciones del sistema de archivos
import shutil                   # Copiar/mover archivos
import random                   # Shuffle para división train/val

# Configuración de matplotlib para modo interactivo
plt.ion()

print("Setup completo.")

**Descomprime las carpetas**

In [ ]:
# Definir rutas según el entorno (Colab o local)
if IN_COLAB:
    # zip_path = f"{BASE_PATH}/OD-WeaponDetection.zip"
    zip_path = f"/content/WeaponDetection.zip"
    extract_to = "/content/datasets"
else:
    # Para entorno local, ajusta estas rutas según tu configuración
    zip_path = f"{BASE_PATH}/OD-WeaponDetection.zip"
    extract_to = f"{BASE_PATH}/datasets"

# Descomprimir solo si no existe o está vacío
if not os.path.exists(extract_to) or len(os.listdir(extract_to)) == 0:
    print("Descomprimiendo dataset (primera vez)...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        files = zip_ref.infolist()
        for f in tqdm(files, desc="Progreso", unit="archivo"):
            zip_ref.extract(f, extract_to)

    print("Completado")
else:
    print("Dataset ya existe. No se vuelve a descomprimir.")

In [ ]:
# Extensiones válidas
IMG_EXT = frozenset({".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"})

def is_image_file(filename):
    return Path(filename).suffix.lower() in IMG_EXT

# ================================
# DEFINICIÓN DE RUTAS
# ================================

if IN_COLAB:
    ROOT_DATASET = Path("/content/datasets")
    OUTPUT_BASE = Path("/content/weapon_dataset")
else:
    ROOT_DATASET = Path(extract_to)
    OUTPUT_BASE = Path(BASE_PATH) / "weapon_dataset"

SOURCE_DIRS = [
    ROOT_DATASET / "Knife_detection",
    ROOT_DATASET / "Pistol_detection",
    ROOT_DATASET / "Weapons and similar handled objects/Sohas_weapon-Detection"
]

FINAL_IMG_DIR = OUTPUT_BASE / "images"
FINAL_ANN_DIR = OUTPUT_BASE / "annotations"

FINAL_IMG_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ANN_DIR.mkdir(parents=True, exist_ok=True)

# ================================
# CONSOLIDACIÓN DE DATASET
# ================================

seen_files = set()

for base_path in SOURCE_DIRS:
    for root, _, files in os.walk(base_path):
        for file in files:
            file_path = Path(root) / file
            ext = file_path.suffix.lower()
            name_no_ext = file_path.stem

            # Imagen
            if ext in IMG_EXT:
                if name_no_ext not in seen_files:
                    seen_files.add(name_no_ext)

                    target_path = FINAL_IMG_DIR / file

                    # Evitar sobreescritura
                    if target_path.exists():
                        target_path = FINAL_IMG_DIR / f"{name_no_ext}_{len(seen_files)}{ext}"

                    shutil.copy(file_path, target_path)

            # XML
            elif ext == ".xml":
                xml_target = FINAL_ANN_DIR / file

                if not xml_target.exists():
                    shutil.copy(file_path, xml_target)

# ================================
# RESULTADOS
# ================================

print(f"Total únicos: {len(seen_files)} imágenes")
print(f"Carpeta final de imágenes: {FINAL_IMG_DIR}")
print(f"Carpeta final de anotaciones: {FINAL_ANN_DIR}")

In [ ]:
# Contar imágenes (usando la misma lógica global)
num_imgs = sum(
    1 for f in FINAL_IMG_DIR.iterdir()
    if f.suffix.lower() in IMG_EXT
)

# Contar anotaciones .xml
num_xml = sum(
    1 for f in FINAL_ANN_DIR.iterdir()
    if f.suffix.lower() == ".xml"
)

print(f"Total de imágenes: {num_imgs}")
print(f"Total de anotaciones (XML): {num_xml}")

**Parseo de anotaciones VOC (XML) PASCAL**

El parseo de anotaciones VOC (XML) se utiliza para leer y extraer la información de las cajas delimitadoras (bounding boxes) y las clases de los objetos anotados en formato PASCAL VOC.
Este proceso convierte esos archivos XML en un formato más usable (como CSV o TXT YOLO) para entrenar modelos de detección de objetos.

In [ ]:
def parse_voc_folder(ann_dir, img_dir):
    rows = []
    missing_count = 0

    # ===========================
    # Índice de imágenes reales en disco
    # clave: nombre sin extensión (stem)
    # valor: nombre completo con extensión
    # ===========================
    image_files = {f.stem: f.name for f in img_dir.iterdir() if f.is_file()}

    for xml_path in sorted(ann_dir.glob('*.xml')):
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # ===========================
        # Nombre del archivo en el XML
        # ===========================
        xml_filename_raw = root.find('filename').text.strip()

        # Obtener nombre sin extensión (stem)
        image_stem = Path(xml_filename_raw).stem

        # ===========================
        # Tamaño de la imagen
        # ===========================
        size = root.find('size')
        W = int(size.find('width').text)
        H = int(size.find('height').text)

        # ===========================
        # Validar existencia de imagen
        # ===========================
        if image_stem not in image_files:
            missing_count += 1
            print(f"⚠ No se encontró imagen para: {image_stem}")
            continue

        # Obtener nombre real (con extensión correcta)
        real_filename = image_files[image_stem]
        img_path = img_dir / real_filename

        # ===========================
        # Extraer objetos (bounding boxes)
        # ===========================
        for obj in root.findall('object'):
            cls = obj.find('name').text.strip().lower()

            bnd = obj.find('bndbox')
            xmin = int(float(bnd.find('xmin').text))
            ymin = int(float(bnd.find('ymin').text))
            xmax = int(float(bnd.find('xmax').text))
            ymax = int(float(bnd.find('ymax').text))

            rows.append({
                'image_path': str(img_path),
                'filename': real_filename,
                'xml_path': str(xml_path),
                'width': W,
                'height': H,
                'xmin': xmin,
                'ymin': ymin,
                'xmax': xmax,
                'ymax': ymax,
                'class': cls
            })

    print(f"[INFO] Imágenes no encontradas: {missing_count}")
    return pd.DataFrame(rows)

**Configuración de directorios y parámetros**

In [ ]:
# Usar las variables de entorno para configurar las rutas
DATASET_DIR = FINAL_IMG_DIR.parent  # Path al directorio weapon_dataset
ANN_DIR = FINAL_ANN_DIR
IMG_DIR = FINAL_IMG_DIR

# Parseo directo del único dataset final
df = parse_voc_folder(ANN_DIR, IMG_DIR)

clases_validas = ["knife", "pistol"]
df = df[df["class"].isin(clases_validas)]

print(f"[OK] Total de registros cargados: {len(df)}")
df.head(5)

Se identificó una discrepancia entre las extensiones de los archivos de imagen referenciados en los archivos XML y las presentes en el sistema de archivos. Los XML utilizan la extensión .jpg en minúsculas, mientras que las imágenes en la carpeta están nombradas con .JPG en mayúsculas, lo que ocasionó que 154 imágenes no fueran encontradas durante el proceso de carga del dataset.

#1.  Análisis exploratorio EDA


Antes de entrenar cualquier modelo de visión por computador, es fundamental realizar un análisis exploratorio del dataset. Esta etapa permite comprender mejor la distribución de clases, la variedad de escenarios presentes en las imágenes, posibles problemas en las anotaciones, así como identificar datos corruptos o desbalanceados. Un buen EDA garantiza que el modelo se entrene sobre información limpia y representativa, lo que mejora su rendimiento y evita errores posteriores.

Este bloque de código extrae información estructural de cada imagen en el dataset. A partir de la lista de rutas de imágenes, obtiene parámetros como la forma completa (alto, ancho y canales), el número de dimensiones (RGB o escala de grises), y las medidas de ancho y alto. Finalmente, organiza estos datos en un DataFrame para facilitar su análisis y validación.

In [ ]:
# ===========================
# Extraer parámetros de imágenes (EDA)
# ===========================

# Lista de rutas de imágenes desde el DataFrame
list_image = df['image_path'].tolist()

# Inicialización de listas para almacenar resultados
data_shape = []   # Forma de la imagen (alto, ancho, canales)
data_dim = []     # Número de dimensiones (2 = grayscale, 3 = RGB)
data_w = []       # Ancho
data_h = []       # Alto

# ===========================
# Recorrido de imágenes
# ===========================

for path in tqdm(list_image):  # tqdm muestra barra de progreso

    try:
        # Abrimos la imagen de forma eficiente
        # 'with' asegura que se libere memoria automáticamente
        with Image.open(path) as img:

            # Extraemos dimensiones sin cargar toda la imagen en memoria
            w, h = img.size  # size devuelve (ancho, alto)

            # Obtenemos el modo de la imagen (RGB, escala de grises, etc.)
            mode = img.mode

            # ===========================
            # Inferencia de dimensiones sin usar numpy
            # ===========================

            if mode == "RGB":
                dimen = 3
                shapes = (h, w, 3)

            elif mode == "L":  # escala de grises
                dimen = 2
                shapes = (h, w)

            else:
                # Fallback: solo si el formato es desconocido
                # Aquí sí convertimos a numpy (más costoso)
                img_array = np.array(img)
                shapes = img_array.shape
                dimen = img_array.ndim

            # Guardamos resultados
            data_w.append(w)
            data_h.append(h)
            data_shape.append(shapes)
            data_dim.append(dimen)

    except Exception as e:
        # Manejo de errores (imágenes corruptas, rutas inválidas, etc.)
        print(f"Error con imagen {path}: {e}")

# ===========================
# Construcción del DataFrame final
# ===========================

data_w_h = pd.DataFrame({
    'filename': list_image,
    'shapes': data_shape,
    'ndim': data_dim,
    'w': data_w,
    'h': data_h
})

**Distribución de imagenes por Alto o Ancho**

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,5))

sns.histplot(data_w_h['w'], bins=40, ax=ax[0], color='steelblue')
ax[0].set_title('Distribución de anchos de imagen (px)')
ax[0].set_xlabel('Ancho (px)')
ax[0].set_ylabel('Frecuencia')

sns.histplot(data_w_h['h'], bins=40, ax=ax[1], color='darkorange')
ax[1].set_title('Distribución de alturas de imagen (px)')
ax[1].set_xlabel('Altura (px)')
ax[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

Ancho: La mayoría de las imágenes están entre 500 y 2000 px, con picos cerca de 1000 y 1500 px. Existen pocos outliers por encima de 3000 px.

Altura: Predominan valores entre 500 y 1500 px, con un pico marcado en 1000 px. También hay casos aislados superiores a 3000 px.

Contamos cuántas imágenes tienen cada ancho (w).

In [ ]:
data_w_h['w'].value_counts()

Contamos cuántas imágenes tienen cada alto (h).

In [ ]:
data_w_h['h'].value_counts()

**Distribución de imagenes por Alto x Ancho**

In [ ]:
data_w_h[['w', 'h']].describe()
sns.scatterplot(data=data_w_h, x='w', y='h')

**Distribución de imagenes por Canales**

In [ ]:
data_w_h['ndim'].value_counts()

Del total de imágenes analizadas, 7.378 presentan tres canales (RGB), mientras que 58 presentan dos canales, lo que sugiere que podrían corresponder a imágenes en escala de grises.

**Distribución de imagenes por Forma**

In [ ]:
data_w_h['shapes'].value_counts()

El tamaño más común es (720, 1280, 3) con 1.453 imágenes, seguido por (120, 160, 3) con 938. Hay combinaciones únicas, desde resoluciones muy pequeñas (120×160) hasta grandes (1920×1281).

Esta alta variabilidad en resoluciones puede afectar el entrenamiento si no se normalizan.

**Distribución de imagenes por Clase**

In [ ]:
df['class'].value_counts()

Observamos un desbalance aproximado de 2 a 1 (2.1 pistolas por cada cuchillo).

Aunque no es un desbalance extremo, en el caso de detección de armas es crítico corregirlo pya que el cuchillo es visualmente más difícil de detectar que una pistola.

Pistola: Tiene una forma volumétrica clara (Forma de L), suele ser oscura y contrasta.

Cuchillo: Es delgado, metálico (refleja luz y se camufla) y dependiendo del ángulo puede verse como una simple línea recta o desaparecer.

Con este desbalance, el modelo optimizará su error ignorando los cuchillos difíciles. Vamos a nivelarlo.

**Distribución de relaciones de aspecto**

In [ ]:
data_w_h['Aspecto'] = data_w_h['w'] / data_w_h['h']

# Distribución de relaciones de aspecto
fig, ax = plt.subplots(figsize=(7,5))
sns.histplot(data_w_h['Aspecto'], bins=40, color='steelblue')
ax.set_title('Distribución de relaciones de aspecto (ancho / alto)')
ax.set_xlabel('Aspect ratio')
ax.set_ylabel('Frecuencia')
plt.show()

# Conteo por categorías de aspect ratio
def aspect_group(r):
    if r < 0.9: return "Vertical"
    elif 0.9 <= r <= 1.1: return "Cuadrada"
    else: return "Horizontal"
data_w_h['Aspecto'] = data_w_h['Aspecto'].apply(aspect_group)

aspect_counts_abs = data_w_h['Aspecto'].value_counts()
aspect_counts_pct = data_w_h['Aspecto'].value_counts(normalize=True) * 100

# Unir ambos resultados en un solo DataFrame
aspect_summary = pd.DataFrame({
    'total': aspect_counts_abs,
    'porcentaje (%)': aspect_counts_pct.round(2)
})

display(aspect_summary)


El 89% de las imágenes son horizontales, con baja proporción vertical (6.39%) y cuadrada (4.51%).

**Verificamos si existen errores en las coordenadas de las cajas delimitadoras (bounding boxes).**

In [ ]:
invalid_boxes = df[
    (df['xmin'] < 0) |
    (df['ymin'] < 0) |
    (df['xmax'] > df['width']) |
    (df['ymax'] > df['height']) |
    (df['xmax'] <= df['xmin']) |
    (df['ymax'] <= df['ymin'])
]

# Relación individual por tipo de error
relacion = (
    df[df['xmin'] >= df['xmax']].shape,
    df[df['ymin'] >= df['ymax']].shape
)

print(f"Bounding boxes inválidos encontrados: {len(invalid_boxes)}")
print(f"Relación de errores (xmin>=xmax, ymin>=ymax): {relacion}")

# --- Mostrar imágenes afectadas ---
if len(invalid_boxes) > 0:
    print("\nImágenes con errores en anotaciones:")
    print(invalid_boxes['filename'].unique())

    print("\nPrimeros bounding boxes inválidos detectados:")
    display(invalid_boxes.head())
else:
    print("No se encontraron bounding boxes inválidos.")


Se identificó una imagen con coordenadas de bounding box inconsistentes. A continuación, se procede a su localización para análisis y corrección

In [ ]:
filenames_invalidos = invalid_boxes['filename'].unique()

# Filtrar en el DataFrame únicamente esos registros
df_filtrado = df[df['filename'].isin(filenames_invalidos)]

print("\nRegistros asociados a esos filenames:")
display(df_filtrado)


Se identificó una imagen con dos anotaciones de bounding box. Se procede a eliminar la anotación inválida para garantizar la consistencia del dataset.

In [ ]:
# Eliminar archivo de imagen
for img_path in df_filtrado["image_path"].unique():
    if os.path.exists(img_path):
        os.remove(img_path)
        print(f"Imagen eliminada: {img_path}")
    else:
        print(f"No existe la imagen: {img_path}")

# Eliminar archivo XML asociado
for xml_path in df_filtrado["xml_path"].unique():
    if os.path.exists(xml_path):
        os.remove(xml_path)
        print(f"XML eliminado: {xml_path}")
    else:
        print(f"No existe el XML: {xml_path}")

# Eliminar los registros del df.
df = df[~df["filename"].isin(df_filtrado["filename"].unique())].reset_index(drop=True)

print("\nRegistros eliminados del DataFrame:", len(df_filtrado))
print("Nuevo tamaño del DataFrame:", df.shape)


Validar que tengan la misma cantidad

In [ ]:
# Contar imágenes únicas
num_imagenes = df['image_path'].nunique()

# Contar XML únicos
num_xml = df['xml_path'].nunique()

print("Imágenes únicas:", num_imagenes)
print("XML únicos:", num_xml)

if num_imagenes == num_xml:
    print("OK: La cantidad coincide")
else:
    print("Diferencia detectada:")
    print("Imágenes únicas:", num_imagenes)
    print("XML únicos:", num_xml)


Validamos nuevamente

In [ ]:
invalid_boxes = df[
    (df['xmin'] < 0) | (df['ymin'] < 0) |
    (df['xmax'] > df['width']) | (df['ymax'] > df['height']) |
    (df['xmax'] <= df['xmin']) | (df['ymax'] <= df['ymin'])
]
relacion = df[df['xmin']>=df['xmax']].shape, df[df['ymin']>=df['ymax']].shape

print(f"Bounding boxes inválidos: {len(invalid_boxes)}, Relacion:{relacion}")

**Distribución del área de los bounding boxes**

In [ ]:
df['bbox_width'] = df['xmax'] - df['xmin']
df['bbox_height'] = df['ymax'] - df['ymin']
df['bbox_area'] = df['bbox_width'] * df['bbox_height']

sns.histplot(df['bbox_area'], bins=50)
plt.title('Distribución del área de los bounding boxes')
plt.show()

**Distribución de objetos por imagen**

In [ ]:
obj_per_img = df.groupby('filename')['class'].count()
sns.histplot(obj_per_img, bins=20)
plt.title('Distribución de objetos por imagen')

La mayoría de las imágenes presentan un solo objeto; pocas contienen múltiples objetos (hasta 10).

**Visualización aleatoria de 5 imagenes**

In [ ]:
sample_rows = df.sample(5)
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    img = cv2.imread(row['image_path'])
    if img is None:
        ax.set_title("Imagen no encontrada")
        ax.axis('off')
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cv2.rectangle(img, (row['xmin'], row['ymin']), (row['xmax'], row['ymax']), (255, 0, 0), 2)
    ax.imshow(img)
    ax.set_title(f"{row['class']}")
    ax.axis('off')

plt.tight_layout()
plt.show()


#2.  Entrenamiendo del modelo

**Estructura necesaria para entrenar modelos YOLO (Ultralytics)**

Para entrenar un modelo YOLO (incluyendo YOLOv8–YOLO11) con Ultralytics, se requiere una estructura de directorios específica y un archivo de configuración (data.yaml).

Cada imagen debe tener un archivo .txt con el mismo nombre.

Los .txt contienen las anotaciones en formato YOLO:
class x_center y_center width height

In [ ]:
# === 1. Carpeta final para YOLO ===
# Definir ruta según el entorno
if IN_COLAB:
    YOLO_DIR = Path("/content/yolo_dataset")
else:
    YOLO_DIR = Path(f"{BASE_PATH}/yolo_dataset")

(IMGS_DIR := YOLO_DIR / "images").mkdir(parents=True, exist_ok=True)
(LABS_DIR := YOLO_DIR / "labels").mkdir(parents=True, exist_ok=True)

# === 2. Mapeo de clases a IDs ===
# class_map = {cls: i for i, cls in enumerate(df["class"].str.lower().unique())}
# print("Class map:", class_map)
CLASS_MAP = {"knife": 0, "pistol": 1}
# print("Class map:", CLASS_MAP)

# === 3. Convertir VOC (xmin,ymin,xmax,ymax) → YOLO (xc,yc,w,h) ===
def voc_to_yolo(row):
    xmin, ymin, xmax, ymax = row["xmin"], row["ymin"], row["xmax"], row["ymax"]
    img_w, img_h = row["width"], row["height"]

    x_center = (xmin + xmax) / 2 / img_w
    y_center = (ymin + ymax) / 2 / img_h
    w = (xmax - xmin) / img_w
    h = (ymax - ymin) / img_h

    cls_id = CLASS_MAP[row["class"].lower()]

    return f"{cls_id} {x_center} {y_center} {w} {h}"

# === 4. Procesar imagen por imagen ===
for filename in df["filename"].unique():

    subset = df[df["filename"] == filename]

    # Obtener ruta de imagen (todas las filas comparten la misma)
    img_path = subset.iloc[0]["image_path"]

    # Copiar imagen
    dst_img = IMGS_DIR / filename
    if os.path.exists(img_path):
        shutil.copy(img_path, dst_img)
    else:
        print(f"⚠ No existe imagen: {img_path}")
        continue

    # Crear archivo .txt
    label_path = LABS_DIR / filename.replace(".jpg", ".txt").replace(".JPG", ".txt").replace(".jpeg", ".txt").replace(".png", ".txt")

    with open(label_path, "w") as f:
        for _, row in subset.iterrows():
            yolo_line = voc_to_yolo(row)
            f.write(yolo_line + "\n")


Revisar que se hayan copiado la totalidad de imágenes y labels:

In [ ]:
# Contar imágenes
num_images = len([f for f in os.listdir(IMGS_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.JPG'))])

# Contar labels
num_labels = len([f for f in os.listdir(LABS_DIR) if f.lower().endswith('.txt')])

print(f"Total de imágenes: {num_images}")
print(f"Total de labels (.txt): {num_labels}")

**Estructura obligatoria del dataset**

```text
yolo_dataset/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

- **images/** contiene todas las imágenes.
- **labels/** contiene los archivos `.txt` con anotaciones YOLO.
- Los nombres de archivo entre *images* y *labels* deben coincidir.
- Las subdivisiones **train/** y **val/** deben tener la misma estructura.

In [ ]:
TRAIN_IMG_DIR = IMGS_DIR / "train"
VAL_IMG_DIR = IMGS_DIR / "val"
TRAIN_LAB_DIR = LABS_DIR / "train"
VAL_LAB_DIR = LABS_DIR / "val"

# Crear carpetas
TRAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)
VAL_IMG_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_LAB_DIR.mkdir(parents=True, exist_ok=True)
VAL_LAB_DIR.mkdir(parents=True, exist_ok=True)

# Lista de imágenes
imgs = sorted([p for p in IMGS_DIR.glob("*") if p.suffix.lower() in [".jpg",".JPG",".jpeg",".png"]])

random.shuffle(imgs)
split = int(len(imgs) * 0.8)
train_imgs = imgs[:split]
val_imgs = imgs[split:]

def move_files(img_list, target_img_dir, target_lab_dir):
    for img_path in img_list:
        label_path = LABS_DIR / (img_path.stem + ".txt")

        # mover imagen
        shutil.move(str(img_path), str(target_img_dir / img_path.name))

        # mover label (si existe)
        if label_path.exists():
            shutil.move(str(label_path), str(target_lab_dir / label_path.name))
        else:
            print(f"⚠ No tiene label: {img_path.name}")


move_files(train_imgs, TRAIN_IMG_DIR, TRAIN_LAB_DIR)
move_files(val_imgs, VAL_IMG_DIR, VAL_LAB_DIR)


Como se pudo observar en el EDA, el dataset presenta un pequeño desbalance, lo que podría ocasionar que el modelo priorice la detección de pistolas sobre cuchillos. Por esta razón, vamos a crear una copia adicional por cada imagen de la clase cuchillos para mejorar el balance.

In [ ]:
def oversample_dataset(img_folder, label_folder, target_class_id, multiplication_factor=2):
    """
    img_folder: Ruta a tus imágenes de TRAIN
    label_folder: Ruta a tus labels de TRAIN (txt)
    target_class_id: ID de la clase a aumentar (ej. 0 para Cuchillo)
    multiplication_factor: Cuántas copias extra crear por cada imagen encontrada
    """

    txt_files = glob(os.path.join(label_folder, "*.txt"))

    print(f"Procesando oversampling para la clase {target_class_id}...")

    count = 0
    for txt_file in tqdm(txt_files):
        has_target_class = False

        # 1. Verificar si el archivo contiene la clase minoritaria
        with open(txt_file, 'r') as f:
            lines = f.readlines()
            for line in lines:
                cls_id = int(line.split()[0])
                if cls_id == target_class_id:
                    has_target_class = True
                    break

        # 2. Si la tiene, duplicar imagen y label
        if has_target_class:
            base_name = os.path.splitext(os.path.basename(txt_file))[0]

            # Buscar la imagen correspondiente (jpg, png o jpeg)
            img_extensions = ['.jpg', '.jpeg', '.png']
            img_file = None
            for ext in img_extensions:
                temp_path = os.path.join(img_folder, base_name + ext)
                if os.path.exists(temp_path):
                    img_file = temp_path
                    img_ext = ext
                    break

            if img_file:
                # Crear copias
                for i in range(multiplication_factor):
                    new_name = f"{base_name}_aug_{i}"

                    # Copiar imagen
                    new_img_path = os.path.join(img_folder, new_name + img_ext)
                    shutil.copy(img_file, new_img_path)

                    # Copiar label
                    new_txt_path = os.path.join(label_folder, new_name + ".txt")
                    shutil.copy(txt_file, new_txt_path)

                    count += 1

    print(f"Se crearon {count} nuevas instancias de imágenes/labels.")

Aplicamos la función en la base de TRAIN.

In [ ]:
class_id_cuchillo = 0

oversample_dataset(
    img_folder=str(TRAIN_IMG_DIR),   # Usar la variable definida anteriormente
    label_folder=str(TRAIN_LAB_DIR), # Usar la variable definida anteriormente
    target_class_id=class_id_cuchillo,
    multiplication_factor=1  # Crear 1 copia extra por cada imagen encontrada
)

Validamos que tenemos la misma cantidad de imagenes y de labels

In [ ]:
def verificar_dataset(base_path):
    # Definimos las rutas asumiendo la estructura estándar de YOLO
    rutas = {
        "Train Images": os.path.join(base_path, 'images/train'),
        "Train Labels": os.path.join(base_path, 'labels/train'),
        "Val Images":   os.path.join(base_path, 'images/val'),
        "Val Labels":   os.path.join(base_path, 'labels/val')
    }

    conteos = {}

    print(f"{'CARPETA':<25} | {'CANTIDAD':<10} | {'TIPO'}")
    print("-" * 50)

    for nombre, ruta in rutas.items():
        if os.path.exists(ruta):
            # Para labels buscamos .txt
            if "Labels" in nombre:
                archivos = glob(os.path.join(ruta, "*.txt"))
                tipo = "TXT"
            # Para imagenes buscamos extensiones comunes
            else:
                extensions = ['*.jpg','*.JPG', '*.jpeg', '*.png', '*.bmp']
                archivos = []
                for ext in extensions:
                    archivos.extend(glob(os.path.join(ruta, ext)))
                tipo = "IMG"

            conteos[nombre] = len(archivos)
            print(f"{nombre:<25} | {len(archivos):<10} | {tipo}")
        else:
            conteos[nombre] = 0
            print(f"{nombre:<25} | {'NO EXISTE':<10} | ❌")

    print("-" * 50)

    # --- VERIFICACIÓN DE PARES ---
    diff_train = conteos["Train Images"] - conteos["Train Labels"]
    diff_val = conteos["Val Images"] - conteos["Val Labels"]

    if diff_train == 0:
        print("TRAIN: Correcto. Cada imagen tiene su label.")
    else:
        print(f"TRAIN: ¡ALERTA! Hay una diferencia de {diff_train} archivos.")
        if diff_train > 0:
            print("Mas imagenes que labels.")
        else:
            print("Mas labels que imágenes.")

    if diff_val == 0:
        print("Cada imagen tiene su label.")
    else:
        print(f"Hay una diferencia de {diff_val} archivos.")

# EJECUTAR VERIFICACIÓN
# Usar la variable YOLO_DIR definida anteriormente
verificar_dataset(str(YOLO_DIR))

**Archivo data.yaml**

In [ ]:
# Usar la variable YOLO_DIR definida anteriormente
yaml_text = f"""
path: {YOLO_DIR}

train: images/train
val: images/val

names:
  0: knife
  1: pistol
"""

with open("weapons.yaml", "w") as f:
    f.write(yaml_text)

**Entrenamiento del modelo YOLO**

In [ ]:
from ultralytics import YOLO
import albumentations as A
import torch

model = YOLO("yolo26s.pt")

print("\n" + "="*50)
print(f"Modelo cargado: {model.ckpt_path}")
print("="*50)


# Detectar automáticamente el device disponible
if torch.cuda.is_available():
    device = 0  # GPU disponible
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'  # Apple  GPU
    print("Apple  GPU (MPS) detectada")
else:
    device = 'cpu'  # CPU como fallback
    print("Usando CPU (sin GPU detectada)")

model.train(
    data="weapons.yaml",
    epochs=25, # YOLO converge rápido, pero necesitamos suficientes épocas para refinar la detección de objetos pequeños, sin embargo por limitacion de recurso lo dejamos en 25.
    imgsz=640, # Resolución de las imagenes
    batch=16,  # Pro limitacion en recurso dejo 16
    project="weapons_training",
    name="yolo_weapons",
    device=device,
    workers=2,        # Colab suele ir bien con 2
    patience=10,      # Early stopping

    # --- HIPERPARÁMETROS DE AUMENTACIÓN ---

    # Geometría (Ángulos difíciles en asaltos)
    mosaic=1.0,         # SIEMPRE activado. Mezcla 4 imágenes. Vital para contexto.
    degrees=15.0,       # Rota la imagen +/- 15°. Las armas nunca están perfectamente rectas en un robo.
    shear=2.5,          # Deforma la perspectiva. Simula mirar desde una cámara en el techo.
    scale=0.5,          # Escala la imagen +/- 50%. Ayuda a detectar armas lejos y cerca.
    fliplr=0.5,         # Espejo horizontal (50% prob). Una pistola se ve igual apuntando a izq o der.

    # Iluminación y Color (Cámaras nocturnas o mala luz)
    hsv_h=0.015,        # Variación pequeña de tono.
    hsv_s=0.7,          # Variación ALTA de saturación. Algunas cámaras saturan mucho o son blanco y negro.
    hsv_v=0.4,          # Variación ALTA de brillo (Valor). Simula luz de día vs. callejón oscuro.

    # # Calidad de Imagen (Lo que diferencia una foto HD de un frame de video)
    # # Nota: Estos valores no vienen por defecto en 0, hay que activarlos para tu caso.
    erasing=0.4,        # (Probabilidad) "Borra" parches aleatorios. Simula oclusiones (mano tapando parte del arma).

    # Salt & Pepper y ruido via Albumentations
    augmentations=[
        A.SaltAndPepper(amount=(0.03, 0.06), p=0.15),  # ← tupla (min, max)
        A.GaussNoise(var_limit=(10, 40), p=0.2),
        A.ImageCompression(quality_lower=55, p=0.2),
    ],

    # 4. Optimización
    optimizer='AdamW',  # Generalmente converge mejor y más estable que SGD para datasets medianos.
    lr0=0.001,          # Learning rate inicial (valor estándar seguro).
    cos_lr=True,        # Decae el learning rate suavemente, ayuda a afinar al final.

    exist_ok=True   # NO crea exp2, usa weapons_training/exp siempre
)

**Guardamos el mejor modelo**

In [ ]:
# Definir ruta según el entorno
if IN_COLAB:
    model_path = "/content/runs/detect/weapons_training/yolo_weapons/weights/best.pt"
else:
    model_path = f"{BASE_PATH}/weapons_training/yolo_weapons/weights/best.pt"

# Cargar tu modelo entrenado
best_model = YOLO(model_path)

# Validación sobre el conjunto de validación
metrics = best_model.val(project="runs", exist_ok=True)

Revisamos metricas de desempeño

In [ ]:
print("Métricas YOLO:")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Graficamos el desempeño del modelo

In [ ]:
# Definir ruta según el entorno
if IN_COLAB:
    val_path = "/content/runs/val"
else:
    val_path = f"{BASE_PATH}/runs/val"

# Obtener todas las imágenes PNG
images = glob(val_path + "/*.png")

# Configurar la cuadrícula
cols = 3  # Número de columnas
rows = (len(images) + cols - 1) // cols  # Calcular filas necesarias

plt.figure(figsize=(15, 5 * rows))

for i, img_path in enumerate(images):
    img = Image.open(img_path)
    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(img_path.split("/")[-1], fontsize=10)

plt.tight_layout()
plt.show()

**Hacemos predicciones en algunas imagenes del dataset de VAL**

In [ ]:
# Usar la variable VAL_IMG_DIR definida anteriormente
val_paths = [str(VAL_IMG_DIR / f) for f in os.listdir(VAL_IMG_DIR)
             if f.lower().endswith((".jpg",".JPG", ".jpeg", ".png"))]

In [ ]:
# Mostrar 10 imágenes de validación
N = 10
images_to_show = val_paths[:N]

# Crear grilla 2x5
fig, axs = plt.subplots(2, 5, figsize=(20, 8))

for ax, img_path in zip(axs.flatten(), images_to_show):

    results = best_model(img_path, conf=0.5)       # predicción
    pred_img = results[0].plot()    # imagen con boxes

    pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)

    ax.imshow(pred_img)
    ax.set_title(img_path.split("/")[-1])
    ax.axis("off")

plt.tight_layout()
plt.show()


**Validamos con algunas imagenes en contexto real, donde los angulos y la luz pueden afectar la predicción**

In [ ]:

# Definir ruta según el entorno
if IN_COLAB:
    val_dir = "/content/datasets/Valid"
else:
    val_dir = f"{BASE_PATH}/datasets/Valid"

# Extensiones permitidas
exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]

# Obtener lista de imágenes
val_paths = []
for e in exts:
    val_paths.extend(glob(f"{val_dir}/{e}"))

print(f"Total imágenes encontradas: {len(val_paths)}")

# Mostrar primeras N imágenes
N = 10
images_to_show = val_paths[:N]

# Crear figura (2 filas x 5 columnas → ajustable)
fig, axs = plt.subplots(2, 5, figsize=(18, 8))

for ax, img_path in zip(axs.flatten(), images_to_show):

    # Predicción con YOLO
    results = best_model(img_path, conf=0.5)

    # Imagen con boxes
    pred_img = results[0].plot()

    # BGR → RGB
    pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)

    # Plot
    ax.imshow(pred_img)
    ax.set_title(img_path.split("/")[-1])
    ax.axis("off")

# Ajustar gráficos
plt.tight_layout()
plt.show()


In [ ]:
import os
import shutil
from pathlib import Path

# ============================================================
# CONFIGURACIÓN — edita el nombre de tu dataset en Kaggle
# ============================================================
KAGGLE_USERNAME = "sebaxtian92"
MODEL_DATASET_NAME = "WeaponDetection"

# Ruta del modelo entrenado
if IN_COLAB:
    model_path = Path("/content/runs/detect/weapons_training/yolo_weapons/weights/best.pt")
else:
    model_path = Path(BASE_PATH) / "weapons_training/yolo11_weapons/weights/best.pt"

# ============================================================
# PREPARAR CARPETA PARA SUBIR
# ============================================================
upload_dir = Path("/content/model_upload")
upload_dir.mkdir(exist_ok=True)

# Copiar el modelo
shutil.copy(model_path, upload_dir / "best.pt")

# Crear metadata requerida por Kaggle
import json
metadata = {
    "title": "Weapon Detection Model - YOLO",
    "id": f"{KAGGLE_USERNAME}/{MODEL_DATASET_NAME}",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(upload_dir / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f)

print(f"Archivos listos para subir:")
for f in upload_dir.iterdir():
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Configura las credenciales de Kaggle
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({
        "username": KAGGLE_USERNAME,
        "key": kaggle_key
    }, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Subir (primera vez: create. Las siguientes: version)
!kaggle datasets create -p /content/model_upload
# Si ya existe el dataset, usa esta línea en vez de la anterior:
# !kaggle datasets version -p /content/model_upload -m "Nuevo entrenamiento"

print("Modelo subido a Kaggle.")
print(f"Disponible en: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{MODEL_DATASET_NAME}")

**Referencias:**

**Documentación oficial Ultralytics**

🔗 Dataset structure (Ultralytics Docs):
https://docs.ultralytics.com/datasets/detect/

🔗 Training guide:
https://docs.ultralytics.com/modes/train/

🔗 YOLO11:
https://docs.ultralytics.com/es/models/yolo11/

@software{yolo11_ultralytics,
  author = {Glenn Jocher and Jing Qiu},
  title = {Ultralytics YOLO11},
  version = {11.0.0},
  year = {2024},
  url = {https://github.com/ultralytics/ultralytics},
  orcid = {0000-0001-5950-6979, 0000-0003-3783-7069},
  license = {AGPL-3.0}
}

**Visualización de IoU (Intersection over Union)**

El IoU mide qué tan bien se superponen las cajas predichas con las cajas reales. Un IoU alto indica que el modelo está prediciendo bounding boxes precisas.

In [ ]:
# Definir ruta según el entorno
if IN_COLAB:
    results_path = "/content/runs/detect/weapons_training/yolo_weapons"
else:
    results_path = f"{BASE_PATH}/weapons_training/yolo_weapons"

# Leer el archivo results.csv que YOLO genera durante el entrenamiento
results_csv = pd.read_csv(f"{results_path}/results.csv")

# Limpiar nombres de columnas (quitar espacios)
results_csv.columns = results_csv.columns.str.strip()

print("Columnas disponibles:")
print(results_csv.columns.tolist())

# Crear visualización de métricas IoU
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Box Loss (train y val) - pérdida relacionada con IoU
axes[0, 0].plot(results_csv['epoch'], results_csv['train/box_loss'], label='Train Box Loss', marker='o')
axes[0, 0].plot(results_csv['epoch'], results_csv['val/box_loss'], label='Val Box Loss', marker='s')
axes[0, 0].set_xlabel('Época')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Box Loss (menor = mejor localización)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. mAP50 - usa IoU threshold de 0.5
if 'metrics/mAP50(B)' in results_csv.columns:
    axes[0, 1].plot(results_csv['epoch'], results_csv['metrics/mAP50(B)'],
                    label='mAP@0.5', color='green', marker='o', linewidth=2)
    axes[0, 1].set_xlabel('Época')
    axes[0, 1].set_ylabel('mAP')
    axes[0, 1].set_title('mAP@IoU=0.5 (mayor = mejor)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_ylim([0, 1])

# 3. mAP50-95 - promedio de IoU de 0.5 a 0.95
if 'metrics/mAP50-95(B)' in results_csv.columns:
    axes[1, 0].plot(results_csv['epoch'], results_csv['metrics/mAP50-95(B)'],
                    label='mAP@0.5:0.95', color='purple', marker='o', linewidth=2)
    axes[1, 0].set_xlabel('Época')
    axes[1, 0].set_ylabel('mAP')
    axes[1, 0].set_title('mAP@IoU=0.5:0.95 (métrica más estricta)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_ylim([0, 1])

# 4. Precision y Recall (relacionados con IoU)
if 'metrics/precision(B)' in results_csv.columns and 'metrics/recall(B)' in results_csv.columns:
    axes[1, 1].plot(results_csv['epoch'], results_csv['metrics/precision(B)'],
                    label='Precision', color='blue', marker='o')
    axes[1, 1].plot(results_csv['epoch'], results_csv['metrics/recall(B)'],
                    label='Recall', color='orange', marker='s')
    axes[1, 1].set_xlabel('Época')
    axes[1, 1].set_ylabel('Valor')
    axes[1, 1].set_title('Precision y Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

# Mostrar métricas finales
print("\n" + "="*50)
print("MÉTRICAS FINALES (última época)")
print("="*50)
last_epoch = results_csv.iloc[-1]
print(f"Box Loss (Val): {last_epoch['val/box_loss']:.4f}")
if 'metrics/mAP50(B)' in results_csv.columns:
    print(f"mAP@0.5: {last_epoch['metrics/mAP50(B)']:.4f}")
if 'metrics/mAP50-95(B)' in results_csv.columns:
    print(f"mAP@0.5:0.95: {last_epoch['metrics/mAP50-95(B)']:.4f}")
if 'metrics/precision(B)' in results_csv.columns:
    print(f"Precision: {last_epoch['metrics/precision(B)']:.4f}")
if 'metrics/recall(B)' in results_csv.columns:
    print(f"Recall: {last_epoch['metrics/recall(B)']:.4f}")

**Interpretación de los Resultados**

Los resultados del entrenamiento muestran un desempeño satisfactorio del modelo YOLO11 para la detección de armas:

**Box Loss (Localización):** La curva de pérdida de localización muestra una disminución progresiva tanto en entrenamiento como en validación, indicando que el modelo está aprendiendo a predecir bounding boxes cada vez más precisas. La convergencia de ambas curvas sin evidencia de sobreajuste (overfitting) sugiere una generalización adecuada.

**mAP@0.5 (Precisión de Detección):** Este métrico evalúa qué tan bien el modelo detecta armas cuando permitimos un IoU de al menos 0.5 (50% de solapamiento entre la caja predicha y la real). Los valores cercanos o superiores a 0.7-0.8 indican que el modelo tiene una buena capacidad para localizar armas en las imágenes, incluso en condiciones desafiantes.

**mAP@0.5:0.95 (Precisión Estricta):** Esta métrica es más exigente ya que promedia el desempeño en múltiples umbrales de IoU (desde 0.5 hasta 0.95). Valores razonables aquí (típicamente entre 0.4-0.6 para detección de objetos complejos) demuestran que el modelo no solo detecta las armas, sino que también predice cajas delimitadoras muy precisas, crucial para aplicaciones de seguridad donde la localización exacta del arma es importante.

**Precision y Recall:** La precision alta indica que cuando el modelo predice que hay un arma, probablemente lo esté haciendo correctamente (pocos falsos positivos). El recall alto significa que el modelo está encontrando la mayoría de las armas presentes en las imágenes (pocos falsos negativos). El balance entre ambas métricas es fundamental: en un sistema de videovigilancia, queremos detectar todas las armas (recall alto) sin generar demasiadas falsas alarmas (precision alta).

**Conclusión:** El modelo YOLO11 entrenado demuestra capacidad robusta para detectar pistolas y cuchillos en diversos contextos. Las mejoras implementadas (balanceo de clases mediante oversampling, data augmentation agresivo, y configuración óptima de hiperparámetros) han resultado en un detector confiable que podría integrarse en sistemas de videovigilancia inteligente para la prevención de delitos.

In [ ]:
# ============================================================
# SEMANA 1 - AUDITORÍA DEL DATASET Y IDENTIFICACIÓN DE GAPS
# ============================================================
# Este análisis evalúa el dataset actual para identificar
# deficiencias en: iluminación, oclusión, ángulos, resolución,
# complejidad de fondo y distribución de clases.
print("Iniciando auditoría del dataset - Semana 1...")
print("=" * 60)

# Fase 1. Auditoría del Dataset e Identificación de Gaps

## Objetivo
Realizar un análisis exhaustivo del dataset actual para identificar deficiencias y áreas de mejora en las siguientes dimensiones:

1. **Estadísticas generales** — Conteo de clases, imágenes, anotaciones
2. **Brillo e iluminación** — Distribución de condiciones luminosas
3. **Contraste** — Evaluar variedad de contraste en las imágenes
4. **Detección de desenfoque (blur)** — Calidad de imagen
5. **Oclusión** — Solapamiento entre bounding boxes
6. **Orientación/ángulos** — Diversidad de orientaciones de las armas
7. **Tamaño de objetos** — Distribución small/medium/large
8. **Complejidad de fondo** — Diversidad de escenarios
9. **Reporte consolidado de gaps** — Hallazgos y recomendaciones

## 1. Resumen estadístico general del dataset

In [ ]:
# ============================================================
# 1. RESUMEN ESTADÍSTICO GENERAL
# ============================================================

print("=" * 60)
print("📊 RESUMEN ESTADÍSTICO DEL DATASET")
print("=" * 60)

# Total de imágenes únicas y anotaciones
total_images = df['filename'].nunique()
total_annotations = len(df)
classes = df['class'].value_counts()

print(f"\n📁 Total de imágenes únicas: {total_images}")
print(f"📝 Total de anotaciones (bounding boxes): {total_annotations}")
print(f"📐 Promedio de objetos por imagen: {total_annotations / total_images:.2f}")
print(f"\n🏷️  Distribución de clases:")
for cls_name, count in classes.items():
    pct = count / total_annotations * 100
    print(f"   • {cls_name}: {count} ({pct:.1f}%)")

# Ratio de desbalance
if len(classes) >= 2:
    ratio = classes.values[0] / classes.values[1]
    print(f"\n⚖️  Ratio de desbalance (mayor/menor): {ratio:.2f}:1")

# Resoluciones
print(f"\n📐 Resoluciones de imagen:")
print(f"   Ancho  — min: {df['width'].min()}, max: {df['width'].max()}, media: {df['width'].mean():.0f}")
print(f"   Alto   — min: {df['height'].min()}, max: {df['height'].max()}, media: {df['height'].mean():.0f}")

# Distribución de clases - gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colors_cls = ['#e74c3c', '#3498db']
classes.plot(kind='bar', ax=axes[0], color=colors_cls[:len(classes)])
axes[0].set_title('Distribución de clases', fontsize=13)
axes[0].set_ylabel('Cantidad de anotaciones')
axes[0].set_xlabel('Clase')
axes[0].tick_params(axis='x', rotation=0)
for i, (cls_name, count) in enumerate(classes.items()):
    axes[0].text(i, count + 50, str(count), ha='center', fontweight='bold')

# Gráfico de objetos por imagen
obj_per_img = df.groupby('filename')['class'].count()
sns.histplot(obj_per_img, bins=range(1, obj_per_img.max() + 2), ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Objetos por imagen', fontsize=13)
axes[1].set_xlabel('Número de objetos')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## 2. Análisis de brillo e iluminación

Evaluamos la distribución de brillo promedio de las imágenes para identificar si el dataset carece de imágenes en condiciones de **baja iluminación** (escenas nocturnas, interiores oscuros) o **sobreexposición**. Un dataset robusto debe cubrir todo el espectro luminoso.

In [ ]:
# ============================================================
# 2. ANÁLISIS DE BRILLO E ILUMINACIÓN
# ============================================================

brightness_values = []
contrast_values = []
blur_values = []
filenames_analyzed = []

unique_images = df.drop_duplicates(subset='filename')

for _, row in tqdm(unique_images.iterrows(), total=len(unique_images), desc="Analizando imágenes"):
    img_path = row['image_path']
    img = cv2.imread(img_path)
    if img is None:
        continue

    # Convertir a escala de grises para análisis
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Brillo: media de intensidad de píxeles (0=negro, 255=blanco)
    brightness = np.mean(gray)
    brightness_values.append(brightness)

    # Contraste: desviación estándar de intensidad
    contrast = np.std(gray)
    contrast_values.append(contrast)

    # Blur: varianza del Laplaciano (menor = más borroso)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    blur_values.append(laplacian_var)

    filenames_analyzed.append(row['filename'])

# Crear DataFrame de análisis
df_quality = pd.DataFrame({
    'filename': filenames_analyzed,
    'brightness': brightness_values,
    'contrast': contrast_values,
    'blur_score': blur_values
})

# Categorizar brillo
def categorize_brightness(b):
    if b < 60:
        return 'Muy oscura'
    elif b < 100:
        return 'Oscura'
    elif b < 160:
        return 'Normal'
    elif b < 200:
        return 'Clara'
    else:
        return 'Sobreexpuesta'

df_quality['brightness_cat'] = df_quality['brightness'].apply(categorize_brightness)

# --- Visualización de brillo ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de brillo
sns.histplot(df_quality['brightness'], bins=50, ax=axes[0], color='goldenrod', edgecolor='white')
axes[0].axvline(x=60, color='red', linestyle='--', alpha=0.7, label='Muy oscura (<60)')
axes[0].axvline(x=100, color='orange', linestyle='--', alpha=0.7, label='Oscura (<100)')
axes[0].axvline(x=200, color='yellow', linestyle='--', alpha=0.7, label='Sobreexpuesta (>200)')
axes[0].set_title('Distribución de brillo (intensidad media)', fontsize=13)
axes[0].set_xlabel('Brillo medio (0-255)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend(fontsize=9)

# Barras por categoría
cat_order = ['Muy oscura', 'Oscura', 'Normal', 'Clara', 'Sobreexpuesta']
cat_colors = ['#2c3e50', '#7f8c8d', '#27ae60', '#f1c40f', '#e74c3c']
brightness_counts = df_quality['brightness_cat'].value_counts().reindex(cat_order, fill_value=0)
brightness_counts.plot(kind='bar', ax=axes[1], color=cat_colors, edgecolor='white')
axes[1].set_title('Categorías de iluminación', fontsize=13)
axes[1].set_ylabel('Cantidad de imágenes')
axes[1].tick_params(axis='x', rotation=45)
for i, count in enumerate(brightness_counts.values):
    pct = count / len(df_quality) * 100
    axes[1].text(i, count + 10, f'{count}\n({pct:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📋 Resumen de iluminación:")
for cat in cat_order:
    count = brightness_counts.get(cat, 0)
    pct = count / len(df_quality) * 100
    print(f"   {cat}: {count} imágenes ({pct:.1f}%)")

## 3. Análisis de contraste

El contraste mide la variabilidad de intensidades en la imagen. Un contraste bajo puede dificultar la detección de armas, especialmente cuando el arma se mimetiza con el fondo. Imágenes de cámaras de vigilancia de baja calidad suelen tener bajo contraste.

In [ ]:
# ============================================================
# 3. ANÁLISIS DE CONTRASTE
# ============================================================

def categorize_contrast(c):
    if c < 30:
        return 'Muy bajo'
    elif c < 50:
        return 'Bajo'
    elif c < 70:
        return 'Normal'
    else:
        return 'Alto'

df_quality['contrast_cat'] = df_quality['contrast'].apply(categorize_contrast)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de contraste
sns.histplot(df_quality['contrast'], bins=50, ax=axes[0], color='teal', edgecolor='white')
axes[0].axvline(x=30, color='red', linestyle='--', alpha=0.7, label='Muy bajo (<30)')
axes[0].axvline(x=50, color='orange', linestyle='--', alpha=0.7, label='Bajo (<50)')
axes[0].set_title('Distribución de contraste (desv. estándar de intensidad)', fontsize=13)
axes[0].set_xlabel('Contraste (std)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend(fontsize=9)

# Barras por categoría
cat_order_c = ['Muy bajo', 'Bajo', 'Normal', 'Alto']
cat_colors_c = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9']
contrast_counts = df_quality['contrast_cat'].value_counts().reindex(cat_order_c, fill_value=0)
contrast_counts.plot(kind='bar', ax=axes[1], color=cat_colors_c, edgecolor='white')
axes[1].set_title('Categorías de contraste', fontsize=13)
axes[1].set_ylabel('Cantidad de imágenes')
axes[1].tick_params(axis='x', rotation=45)
for i, count in enumerate(contrast_counts.values):
    pct = count / len(df_quality) * 100
    axes[1].text(i, count + 10, f'{count}\n({pct:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📋 Resumen de contraste:")
for cat in cat_order_c:
    count = contrast_counts.get(cat, 0)
    pct = count / len(df_quality) * 100
    print(f"   {cat}: {count} imágenes ({pct:.1f}%)")

## 4. Detección de desenfoque (Blur)

La varianza del Laplaciano es un indicador de nitidez: valores bajos indican imágenes borrosas. En escenarios de vigilancia real, el movimiento y la baja calidad de las cámaras producen desenfoque. Necesitamos verificar cuántas imágenes borrosas existen actualmente en el dataset.

In [ ]:
# ============================================================
# 4. DETECCIÓN DE DESENFOQUE (BLUR)
# ============================================================

def categorize_blur(b):
    if b < 50:
        return 'Muy borrosa'
    elif b < 200:
        return 'Borrosa'
    elif b < 500:
        return 'Aceptable'
    else:
        return 'Nítida'

df_quality['blur_cat'] = df_quality['blur_score'].apply(categorize_blur)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de blur (escala log para mejor visualización)
sns.histplot(df_quality['blur_score'].clip(upper=5000), bins=60, ax=axes[0], color='mediumpurple', edgecolor='white')
axes[0].axvline(x=50, color='red', linestyle='--', alpha=0.7, label='Muy borrosa (<50)')
axes[0].axvline(x=200, color='orange', linestyle='--', alpha=0.7, label='Borrosa (<200)')
axes[0].set_title('Distribución de nitidez (varianza Laplaciano)', fontsize=13)
axes[0].set_xlabel('Blur Score (mayor = más nítida)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend(fontsize=9)

# Barras por categoría
cat_order_b = ['Muy borrosa', 'Borrosa', 'Aceptable', 'Nítida']
cat_colors_b = ['#e74c3c', '#e67e22', '#f1c40f', '#27ae60']
blur_counts = df_quality['blur_cat'].value_counts().reindex(cat_order_b, fill_value=0)
blur_counts.plot(kind='bar', ax=axes[1], color=cat_colors_b, edgecolor='white')
axes[1].set_title('Categorías de nitidez', fontsize=13)
axes[1].set_ylabel('Cantidad de imágenes')
axes[1].tick_params(axis='x', rotation=45)
for i, count in enumerate(blur_counts.values):
    pct = count / len(df_quality) * 100
    axes[1].text(i, count + 10, f'{count}\n({pct:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📋 Resumen de nitidez:")
for cat in cat_order_b:
    count = blur_counts.get(cat, 0)
    pct = count / len(df_quality) * 100
    print(f"   {cat}: {count} imágenes ({pct:.1f}%)")

## 5. Análisis de oclusión

Evaluamos el grado de solapamiento (IoU) entre bounding boxes dentro de la misma imagen. Cuando múltiples objetos se superponen, se simula oclusión parcial. También analizamos la proporción del arma visible (ratio bbox/imagen) para detectar oclusiones indirectas.

In [ ]:
# ============================================================
# 5. ANÁLISIS DE OCLUSIÓN
# ============================================================

def compute_iou(box1, box2):
    """Calcula IoU entre dos bounding boxes [xmin, ymin, xmax, ymax]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area

    return inter_area / union_area if union_area > 0 else 0

# Analizar imágenes con múltiples objetos
multi_obj_images = df.groupby('filename').filter(lambda x: len(x) > 1)
occlusion_data = []

for filename, group in multi_obj_images.groupby('filename'):
    boxes = group[['xmin', 'ymin', 'xmax', 'ymax']].values
    max_iou = 0
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            iou = compute_iou(boxes[i], boxes[j])
            max_iou = max(max_iou, iou)
    occlusion_data.append({'filename': filename, 'max_iou': max_iou, 'num_objects': len(group)})

df_occlusion = pd.DataFrame(occlusion_data)

# Calcular ratio bbox/imagen (qué porcentaje de la imagen ocupa el arma)
df['bbox_area'] = (df['xmax'] - df['xmin']) * (df['ymax'] - df['ymin'])
df['img_area'] = df['width'] * df['height']
df['bbox_ratio'] = df['bbox_area'] / df['img_area']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# IoU entre objetos en la misma imagen
if len(df_occlusion) > 0:
    sns.histplot(df_occlusion['max_iou'], bins=30, ax=axes[0], color='crimson', edgecolor='white')
    axes[0].set_title(f'IoU máximo entre objetos\n({len(df_occlusion)} imágenes multi-objeto)', fontsize=12)
    axes[0].set_xlabel('IoU máximo')
    axes[0].set_ylabel('Frecuencia')
    occluded = (df_occlusion['max_iou'] > 0.1).sum()
    axes[0].axvline(x=0.1, color='orange', linestyle='--', label=f'Oclusión >10%: {occluded} imgs')
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, 'No hay imágenes\nmulti-objeto', ha='center', va='center', fontsize=14)
    axes[0].set_title('IoU entre objetos')

# Distribución del ratio bbox/imagen
sns.histplot(df['bbox_ratio'], bins=50, ax=axes[1], color='darkcyan', edgecolor='white')
axes[1].set_title('Ratio área-bbox / área-imagen', fontsize=12)
axes[1].set_xlabel('Ratio (0=muy pequeño, 1=toda la imagen)')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(x=0.01, color='red', linestyle='--', alpha=0.7, label='Muy pequeño (<1%)')
axes[1].legend()

# Objetos por clase según tamaño relativo
def size_category(ratio):
    if ratio < 0.01:
        return 'Muy pequeño (<1%)'
    elif ratio < 0.05:
        return 'Pequeño (1-5%)'
    elif ratio < 0.15:
        return 'Mediano (5-15%)'
    elif ratio < 0.40:
        return 'Grande (15-40%)'
    else:
        return 'Muy grande (>40%)'

df['size_cat'] = df['bbox_ratio'].apply(size_category)
size_class = df.groupby(['class', 'size_cat']).size().unstack(fill_value=0)
size_order = ['Muy pequeño (<1%)', 'Pequeño (1-5%)', 'Mediano (5-15%)', 'Grande (15-40%)', 'Muy grande (>40%)']
size_class = size_class.reindex(columns=size_order, fill_value=0)
size_class.plot(kind='bar', ax=axes[2], edgecolor='white')
axes[2].set_title('Tamaño relativo del objeto por clase', fontsize=12)
axes[2].set_ylabel('Cantidad')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

print("\n📋 Resumen de oclusión:")
print(f"   Imágenes con un solo objeto: {df['filename'].nunique() - len(df_occlusion)}")
print(f"   Imágenes con múltiples objetos: {len(df_occlusion)}")
if len(df_occlusion) > 0:
    print(f"   Imágenes con oclusión significativa (IoU > 0.1): {(df_occlusion['max_iou'] > 0.1).sum()}")

print(f"\n📋 Distribución por tamaño relativo del arma:")
for cat in size_order:
    count = (df['size_cat'] == cat).sum()
    pct = count / len(df) * 100
    print(f"   {cat}: {count} ({pct:.1f}%)")

## 6. Análisis de orientación y ángulos de los bounding boxes

La relación de aspecto de los bounding boxes nos indica la orientación predominante de las armas. Un dataset diverso debe contener armas en orientación horizontal, vertical y diagonal. Analizamos los aspect ratios de los bounding boxes como proxy de la orientación.

In [ ]:
# ============================================================
# 6. ANÁLISIS DE ORIENTACIÓN Y ÁNGULOS
# ============================================================

# Aspect ratio de los bounding boxes (ancho_bbox / alto_bbox)
df['bbox_width'] = df['xmax'] - df['xmin']
df['bbox_height'] = df['ymax'] - df['ymin']
df['bbox_aspect_ratio'] = df['bbox_width'] / df['bbox_height'].replace(0, 1)

# Categorizar orientación basada en aspect ratio
def orient_category(ar):
    if ar < 0.5:
        return 'Vertical fuerte'
    elif ar < 0.85:
        return 'Vertical leve'
    elif ar <= 1.15:
        return 'Cuadrado/Diagonal'
    elif ar <= 2.0:
        return 'Horizontal leve'
    else:
        return 'Horizontal fuerte'

df['orientation'] = df['bbox_aspect_ratio'].apply(orient_category)

# Posición del centro del bbox dentro de la imagen
df['center_x_rel'] = ((df['xmin'] + df['xmax']) / 2) / df['width']
df['center_y_rel'] = ((df['ymin'] + df['ymax']) / 2) / df['height']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Aspect ratio por clase
for cls in df['class'].unique():
    subset = df[df['class'] == cls]
    sns.histplot(subset['bbox_aspect_ratio'].clip(upper=5), bins=40, ax=axes[0],
                 label=cls, alpha=0.6, edgecolor='white')
axes[0].set_title('Aspect ratio de bounding boxes por clase', fontsize=12)
axes[0].set_xlabel('Aspect ratio (ancho/alto)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].axvline(x=1, color='gray', linestyle='--', alpha=0.5)

# 2. Distribución de orientación
orient_order = ['Vertical fuerte', 'Vertical leve', 'Cuadrado/Diagonal', 'Horizontal leve', 'Horizontal fuerte']
orient_counts = df['orientation'].value_counts().reindex(orient_order, fill_value=0)
orient_colors = ['#3498db', '#5dade2', '#95a5a6', '#e67e22', '#e74c3c']
orient_counts.plot(kind='bar', ax=axes[1], color=orient_colors, edgecolor='white')
axes[1].set_title('Distribución de orientaciones', fontsize=12)
axes[1].set_ylabel('Cantidad')
axes[1].tick_params(axis='x', rotation=45)
for i, count in enumerate(orient_counts.values):
    pct = count / len(df) * 100
    axes[1].text(i, count + 20, f'{pct:.1f}%', ha='center', fontsize=9)

# 3. Posición del centro de bbox en la imagen
axes[2].scatter(df['center_x_rel'], df['center_y_rel'], alpha=0.15, s=3, c='steelblue')
axes[2].set_title('Posición de objetos en la imagen', fontsize=12)
axes[2].set_xlabel('Posición X relativa (0=izq, 1=der)')
axes[2].set_ylabel('Posición Y relativa (0=arriba, 1=abajo)')
axes[2].set_xlim(0, 1)
axes[2].set_ylim(1, 0)  # Invertir eje Y para simular coordenadas de imagen
axes[2].set_aspect('equal')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📋 Resumen de orientación:")
for orient in orient_order:
    count = orient_counts.get(orient, 0)
    pct = count / len(df) * 100
    print(f"   {orient}: {count} ({pct:.1f}%)")

# Análisis de orientación por clase
print("\n📋 Orientación por clase:")
orient_by_class = pd.crosstab(df['class'], df['orientation'], normalize='index') * 100
print(orient_by_class.round(1).to_string())

## 7. Análisis de complejidad de fondo

La complejidad del fondo se evalúa midiendo la cantidad de bordes/detalle en la región de la imagen **fuera** del bounding box. Fondos complejos (como multitudes, calles, tiendas) dificultan la detección en comparación con fondos simples (uniforme, estudio).

In [ ]:
# ============================================================
# 7. ANÁLISIS DE COMPLEJIDAD DE FONDO
# ============================================================

bg_complexity = []

sample_for_bg = unique_images.sample(min(500, len(unique_images)), random_state=42)

for _, row in tqdm(sample_for_bg.iterrows(), total=len(sample_for_bg), desc="Analizando fondos"):
    img = cv2.imread(row['image_path'])
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Crear máscara excluyendo bounding boxes
    mask = np.ones_like(gray, dtype=bool)
    img_annotations = df[df['filename'] == row['filename']]
    for _, ann in img_annotations.iterrows():
        y1 = max(0, int(ann['ymin']))
        y2 = min(gray.shape[0], int(ann['ymax']))
        x1 = max(0, int(ann['xmin']))
        x2 = min(gray.shape[1], int(ann['xmax']))
        mask[y1:y2, x1:x2] = False

    # Complejidad del fondo = densidad de bordes fuera de los bboxes
    edges = cv2.Canny(gray, 50, 150)
    bg_edges = edges[mask]
    edge_density = np.mean(bg_edges) if len(bg_edges) > 0 else 0

    bg_complexity.append({
        'filename': row['filename'],
        'edge_density': edge_density
    })

df_bg = pd.DataFrame(bg_complexity)

def bg_category(density):
    if density < 5:
        return 'Simple'
    elif density < 15:
        return 'Moderado'
    elif density < 30:
        return 'Complejo'
    else:
        return 'Muy complejo'

df_bg['bg_cat'] = df_bg['edge_density'].apply(bg_category)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de complejidad
sns.histplot(df_bg['edge_density'], bins=40, ax=axes[0], color='darkolivegreen', edgecolor='white')
axes[0].set_title('Distribución de complejidad de fondo\n(densidad de bordes)', fontsize=12)
axes[0].set_xlabel('Densidad de bordes en fondo')
axes[0].set_ylabel('Frecuencia')

# Barras por categoría
cat_order_bg = ['Simple', 'Moderado', 'Complejo', 'Muy complejo']
cat_colors_bg = ['#27ae60', '#f1c40f', '#e67e22', '#e74c3c']
bg_counts = df_bg['bg_cat'].value_counts().reindex(cat_order_bg, fill_value=0)
bg_counts.plot(kind='bar', ax=axes[1], color=cat_colors_bg, edgecolor='white')
axes[1].set_title('Categorías de complejidad de fondo', fontsize=12)
axes[1].set_ylabel('Cantidad de imágenes (muestra)')
axes[1].tick_params(axis='x', rotation=45)
for i, count in enumerate(bg_counts.values):
    pct = count / len(df_bg) * 100
    axes[1].text(i, count + 3, f'{count}\n({pct:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n📋 Resumen de complejidad de fondo (muestra de {len(df_bg)} imágenes):")
for cat in cat_order_bg:
    count = bg_counts.get(cat, 0)
    pct = count / len(df_bg) * 100
    print(f"   {cat}: {count} ({pct:.1f}%)")

## 8. Análisis de resolución por categoría COCO

Clasificamos los objetos según los criterios de tamaño de COCO: **small** (<32x32 px), **medium** (32-96 px), **large** (>96x96 px). Esto nos indica si el modelo tendrá dificultades con objetos pequeños (armas a distancia).

In [ ]:
# ============================================================
# 8. TAMAÑO DE OBJETOS (CRITERIO COCO)
# ============================================================

def coco_size(area):
    if area < 32**2:
        return 'Small (<32²)'
    elif area < 96**2:
        return 'Medium (32²-96²)'
    else:
        return 'Large (>96²)'

df['coco_size'] = df['bbox_area'].apply(coco_size)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución general
coco_order = ['Small (<32²)', 'Medium (32²-96²)', 'Large (>96²)']
coco_colors = ['#e74c3c', '#f39c12', '#27ae60']
coco_counts = df['coco_size'].value_counts().reindex(coco_order, fill_value=0)
coco_counts.plot(kind='bar', ax=axes[0], color=coco_colors, edgecolor='white')
axes[0].set_title('Tamaño de objetos (criterio COCO)', fontsize=13)
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=0)
for i, count in enumerate(coco_counts.values):
    pct = count / len(df) * 100
    axes[0].text(i, count + 30, f'{count}\n({pct:.1f}%)', ha='center', fontsize=9)

# Por clase
coco_by_class = pd.crosstab(df['class'], df['coco_size'], normalize='index') * 100
coco_by_class = coco_by_class.reindex(columns=coco_order, fill_value=0)
coco_by_class.plot(kind='bar', ax=axes[1], color=coco_colors, edgecolor='white')
axes[1].set_title('Tamaño por clase (%)', fontsize=13)
axes[1].set_ylabel('Porcentaje')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\n📋 Resumen de tamaños (criterio COCO):")
for cat in coco_order:
    count = coco_counts.get(cat, 0)
    pct = count / len(df) * 100
    print(f"   {cat}: {count} ({pct:.1f}%)")

## 9. Mapa de calor: posición de objetos en la imagen

Visualizamos dónde tienden a aparecer las armas dentro de las imágenes. Esto revela sesgos posicionales (p.ej., armas siempre centradas) que pueden limitar la detección en escenarios reales donde las armas pueden aparecer en cualquier zona del encuadre.

In [ ]:
# ============================================================
# 9. MAPA DE CALOR DE POSICIÓN DE OBJETOS
# ============================================================

# Crear mapa de calor normalizado
heatmap = np.zeros((100, 100))

for _, row in df.iterrows():
    cx = int(((row['xmin'] + row['xmax']) / 2 / row['width']) * 99)
    cy = int(((row['ymin'] + row['ymax']) / 2 / row['height']) * 99)
    cx = np.clip(cx, 0, 99)
    cy = np.clip(cy, 0, 99)

    # Tamaño relativo del bbox
    bw = int((row['bbox_width'] / row['width']) * 50)
    bh = int((row['bbox_height'] / row['height']) * 50)
    bw = max(1, bw)
    bh = max(1, bh)

    y1 = max(0, cy - bh // 2)
    y2 = min(99, cy + bh // 2)
    x1 = max(0, cx - bw // 2)
    x2 = min(99, cx + bw // 2)
    heatmap[y1:y2+1, x1:x2+1] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Mapa de calor general
im = axes[0].imshow(heatmap, cmap='hot', interpolation='bilinear')
axes[0].set_title('Mapa de calor: posición de armas\n(todas las clases)', fontsize=12)
axes[0].set_xlabel('Posición X normalizada')
axes[0].set_ylabel('Posición Y normalizada')
plt.colorbar(im, ax=axes[0], shrink=0.8, label='Densidad')

# Mapa de calor por clase
heatmaps_cls = {}
for cls in df['class'].unique():
    hm = np.zeros((100, 100))
    cls_df = df[df['class'] == cls]
    for _, row in cls_df.iterrows():
        cx = int(((row['xmin'] + row['xmax']) / 2 / row['width']) * 99)
        cy = int(((row['ymin'] + row['ymax']) / 2 / row['height']) * 99)
        cx = np.clip(cx, 0, 99)
        cy = np.clip(cy, 0, 99)
        hm[max(0,cy-2):min(99,cy+3), max(0,cx-2):min(99,cx+3)] += 1
    heatmaps_cls[cls] = hm

# Overlay ambas clases en un solo mapa con diferentes canales
combined = np.zeros((100, 100, 3))
classes_list = list(heatmaps_cls.keys())
if len(classes_list) >= 2:
    # Normalizar
    h1 = heatmaps_cls[classes_list[0]]
    h2 = heatmaps_cls[classes_list[1]]
    if h1.max() > 0:
        h1 = h1 / h1.max()
    if h2.max() > 0:
        h2 = h2 / h2.max()
    combined[:,:,0] = h1  # Rojo = clase 1
    combined[:,:,2] = h2  # Azul = clase 2

    axes[1].imshow(combined, interpolation='bilinear')
    axes[1].set_title(f'Distribución espacial por clase\nRojo={classes_list[0]} | Azul={classes_list[1]}', fontsize=12)
else:
    axes[1].imshow(heatmaps_cls[classes_list[0]], cmap='hot', interpolation='bilinear')
    axes[1].set_title(f'Distribución espacial: {classes_list[0]}', fontsize=12)

axes[1].set_xlabel('Posición X normalizada')
axes[1].set_ylabel('Posición Y normalizada')

plt.tight_layout()
plt.show()

## 10. Reporte consolidado de gaps e identificación de deficiencias

Consolidamos todos los hallazgos en un reporte de gaps accionable, con recomendaciones concretas para la Semana 2 (expansión del dataset).

In [ ]:
# ============================================================
# 10. REPORTE CONSOLIDADO DE GAPS
# ============================================================

print("=" * 70)
print("📋 REPORTE CONSOLIDADO: AUDITORÍA DEL DATASET Y GAPS IDENTIFICADOS")
print("=" * 70)
print(f"Fecha: Semana 1 del cronograma")
print(f"Dataset: DaSCI Weapon Detection (pistol + knife)")
print(f"Total imágenes: {df['filename'].nunique()} | Total anotaciones: {len(df)}")
print()

# --- Recopilar gaps ---
gaps = []
recommendations = []

# 1. Balance de clases
ratio = classes.values[0] / classes.values[1] if len(classes) >= 2 else 1
if ratio > 1.5:
    gaps.append(f"⚠️  DESBALANCE DE CLASES: ratio {ratio:.1f}:1 ({classes.index[0]} vs {classes.index[1]})")
    recommendations.append("→ Aplicar oversampling, data augmentation dirigido, o recopilar más imágenes de la clase minoritaria")

# 2. Iluminación
dark_pct = (brightness_counts.get('Muy oscura', 0) + brightness_counts.get('Oscura', 0)) / len(df_quality) * 100
bright_pct = brightness_counts.get('Sobreexpuesta', 0) / len(df_quality) * 100
if dark_pct < 15:
    gaps.append(f"⚠️  DÉFICIT DE IMÁGENES OSCURAS: solo {dark_pct:.1f}% del dataset es oscuro/muy oscuro")
    recommendations.append("→ Recopilar imágenes nocturnas, interiores con poca luz, frames de cámaras CCTV nocturnas")
if bright_pct < 5:
    gaps.append(f"⚠️  POCAS IMÁGENES SOBREEXPUESTAS: {bright_pct:.1f}%")
    recommendations.append("→ Incluir imágenes exteriores con luz directa o reflejos")

# 3. Blur
blurry_pct = (blur_counts.get('Muy borrosa', 0) + blur_counts.get('Borrosa', 0)) / len(df_quality) * 100
if blurry_pct < 10:
    gaps.append(f"⚠️  POCAS IMÁGENES BORROSAS: {blurry_pct:.1f}% — el modelo puede fallar con video de baja calidad")
    recommendations.append("→ Agregar imágenes con motion blur, frames de video comprimido, zoom digital")

# 4. Contraste
low_contrast_pct = (contrast_counts.get('Muy bajo', 0) + contrast_counts.get('Bajo', 0)) / len(df_quality) * 100
if low_contrast_pct < 10:
    gaps.append(f"⚠️  POCAS IMÁGENES DE BAJO CONTRASTE: {low_contrast_pct:.1f}%")
    recommendations.append("→ Incluir imágenes tipo CCTV, niebla, lluvia, o cámaras analógicas")

# 5. Oclusión
single_obj_pct = (df['filename'].nunique() - len(df_occlusion)) / df['filename'].nunique() * 100
if single_obj_pct > 80:
    gaps.append(f"⚠️  PREDOMINIO DE OBJETO ÚNICO: {single_obj_pct:.1f}% de imágenes tienen un solo objeto")
    recommendations.append("→ Agregar imágenes con múltiples objetos, armas parcialmente ocultas por manos/cuerpos")

# 6. Objetos pequeños
small_pct = coco_counts.get('Small (<32²)', 0) / len(df) * 100
if small_pct < 10:
    gaps.append(f"⚠️  POCOS OBJETOS PEQUEÑOS (según COCO): {small_pct:.1f}% — detección a distancia limitada")
    recommendations.append("→ Incluir imágenes tomadas a distancia, tipo vigilancia gran angular")

# 7. Orientación
orient_diversity = orient_counts.min() / orient_counts.max() if orient_counts.max() > 0 else 0
if orient_diversity < 0.15:
    dominant = orient_counts.idxmax()
    gaps.append(f"⚠️  BAJA DIVERSIDAD DE ORIENTACIÓN: predomina '{dominant}' ({orient_counts.max()/len(df)*100:.1f}%)")
    recommendations.append("→ Incluir armas en ángulos variados: apuntando arriba/abajo, diagonal, en el piso")

# 8. Complejidad de fondo
simple_bg_pct = bg_counts.get('Simple', 0) / len(df_bg) * 100
if simple_bg_pct > 40:
    gaps.append(f"⚠️  EXCESO DE FONDOS SIMPLES: {simple_bg_pct:.1f}% — puede causar falsos positivos en escenas reales")
    recommendations.append("→ Agregar imágenes en calles, transporte público, tiendas, multitudes")

# --- Imprimir reporte ---
print("🔴 GAPS IDENTIFICADOS:")
print("-" * 50)
if len(gaps) == 0:
    print("   ✅ No se identificaron gaps significativos")
else:
    for i, gap in enumerate(gaps, 1):
        print(f"   {i}. {gap}")

print()
print("💡 RECOMENDACIONES PARA SEMANA 2:")
print("-" * 50)
if len(recommendations) == 0:
    print("   ✅ El dataset parece bien balanceado y diverso")
else:
    for i, rec in enumerate(recommendations, 1):
        print(f"   {i}. {rec}")

print()
print("=" * 70)
print("📊 RESUMEN NUMÉRICO")
print("=" * 70)
print(f"   Clases:                    {len(classes)} ({', '.join(classes.index)})")
print(f"   Imágenes únicas:           {df['filename'].nunique()}")
print(f"   Anotaciones totales:       {len(df)}")
print(f"   Ratio desbalance:          {ratio:.2f}:1")
print(f"   % imágenes oscuras:        {dark_pct:.1f}%")
print(f"   % imágenes borrosas:       {blurry_pct:.1f}%")
print(f"   % bajo contraste:          {low_contrast_pct:.1f}%")
print(f"   % objetos small (COCO):    {small_pct:.1f}%")
print(f"   % un solo objeto/imagen:   {single_obj_pct:.1f}%")
print(f"   % fondos simples:          {simple_bg_pct:.1f}%")
print(f"   Gaps identificados:        {len(gaps)}")
print("=" * 70)

## 11. Visualización de casos extremos

Mostramos ejemplos concretos de las imágenes más oscuras, más borrosas y con menor contraste del dataset. Esto permite validar visualmente los gaps identificados numéricamente.

In [ ]:
# ============================================================
# 11. VISUALIZACIÓN DE CASOS EXTREMOS
# ============================================================

# Unir df_quality con df para tener las rutas de imagen
df_quality_merged = df_quality.merge(
    df[['filename', 'image_path']].drop_duplicates(subset='filename'),
    on='filename',
    how='left'
)

fig, axes = plt.subplots(3, 5, figsize=(20, 12))

# --- Fila 1: 5 imágenes más oscuras ---
darkest = df_quality_merged.nsmallest(5, 'brightness')
for ax, (_, row) in zip(axes[0], darkest.iterrows()):
    img = cv2.imread(row['image_path'])
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"Brillo: {row['brightness']:.0f}", fontsize=10, color='red')
    ax.axis('off')
axes[0][0].set_ylabel('Más oscuras', fontsize=12, fontweight='bold')

# --- Fila 2: 5 imágenes más borrosas ---
blurriest = df_quality_merged.nsmallest(5, 'blur_score')
for ax, (_, row) in zip(axes[1], blurriest.iterrows()):
    img = cv2.imread(row['image_path'])
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"Blur: {row['blur_score']:.0f}", fontsize=10, color='orange')
    ax.axis('off')
axes[1][0].set_ylabel('Más borrosas', fontsize=12, fontweight='bold')

# --- Fila 3: 5 imágenes con menor contraste ---
lowest_contrast = df_quality_merged.nsmallest(5, 'contrast')
for ax, (_, row) in zip(axes[2], lowest_contrast.iterrows()):
    img = cv2.imread(row['image_path'])
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"Contraste: {row['contrast']:.0f}", fontsize=10, color='purple')
    ax.axis('off')
axes[2][0].set_ylabel('Menor contraste', fontsize=12, fontweight='bold')

plt.suptitle('Casos extremos del dataset (candidatos a problemáticos)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# (Siguiente paso) Fase 2. Expansión del dataset según los gaps identificados


## Objetivo
Ampliar el dataset actual (~5,000 imágenes de pistolas y cuchillos) con imágenes adicionales provenientes de fuentes públicas, priorizando la cobertura de los **gaps identificados** en la auditoría: iluminación baja, desenfoque, diversidad de ángulos, fondos complejos y oclusión.

---

## Fuentes especializadas en detección de armas

| # | Fuente | Descripción | Formato disponible | URL |
|---|--------|-------------|--------------------|-----|
| 1 | **Roboflow Universe** | Múltiples datasets comunitarios de armas ya anotados. Buscar: "weapon detection", "gun detection", "knife detection". Exportables directamente en formato YOLO. | YOLO, Pascal VOC, COCO | https://universe.roboflow.com/search?q=weapon+detection |
| 2 | **DaSCI Open Data** | Fuente actual del proyecto. Verificar si existen subconjuntos adicionales no descargados (Sohas completo, otras categorías). | Pascal VOC (XML) | https://dasci.es/opendata/deteccion-de-armas-open-data/ |
| 3 | **Kaggle** | Datasets comunitarios variados de armas. Buscar: "weapon detection dataset", "gun dataset", "knife detection". | Variado (YOLO, CSV, XML) | https://www.kaggle.com/search?q=weapon+detection+dataset |
| 4 | **Open Images V7 (Google)** | Dataset masivo con las clases "Knife", "Handgun" y "Rifle" con bounding boxes verificados. Se pueden filtrar y descargar solo las clases de interés. | OID format → convertible a YOLO | https://storage.googleapis.com/openimages/web/index.html |

---

## Fuentes complementarias (escenas de vigilancia realistas)

| # | Fuente | Descripción | Uso recomendado | URL |
|---|--------|-------------|-----------------|-----|
| 5 | **UCF Crime Dataset** | Videos de vigilancia reales con escenas de robos y asaltos. Permite extraer frames con condiciones reales: baja iluminación, ángulos de cámaras CCTV, desenfoque por movimiento. | Extraer frames para cubrir gaps de iluminación y blur | https://www.crcv.ucf.edu/projects/real-world/ |
| 6 | **COCO Dataset** | Contiene la clase "knife" entre sus 80 categorías. Imágenes con contextos variados y fondos complejos. | Extraer imágenes con cuchillos en escenas cotidianas | https://cocodataset.org/#download |

---

## Datasets recomendados en Roboflow (listos para YOLO)

Estos datasets pueden exportarse directamente en formato YOLOv8/YOLO11 desde Roboflow:

- **"Guns Object Detection"** — ~5,000 imágenes de armas de fuego
- **"Weapon Detection"** — pistolas, cuchillos y rifles con anotaciones
- **"Knife Detection Dataset"** — enfocado en armas blancas en contextos variados


## Estrategia de integración

1. **Prioridad 1 — Roboflow Universe**: descargar 2-3 datasets anotados en formato YOLO y combinarlos con el dataset actual.
2. **Prioridad 2 — Open Images V7**: extraer imágenes de "Knife" y "Handgun" para cubrir gaps de iluminación y diversidad de contexto.
3. **Prioridad 3 — UCF Crime**: extraer frames de videos de vigilancia para obtener imágenes realistas (borrosas, oscuras, ángulos cenitales).
4. **Unificación**: convertir todas las anotaciones al formato YOLO (`class x_center y_center width height`) antes de integrar al dataset principal.

**Meta**: alcanzar ~12,000 imágenes anotadas cubriendo las deficiencias identificadas en la auditoría.